# Exercise 5: Checking a community-led project plan

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Endangered Languages and Ethics · version 0.1.0*

> Draft a community-led preservation project with alternative deliverables, budget, individual and collective permissions, withdrawal limits and maintenance. This is a desk exercise; contacting or recording a vulnerable community is not required.

**Setup.** `pip install requests huggingface_hub fasttext-wheel transformers torch sentencepiece "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this, and Exercise 1 also calls `git`. Data come from commit- or tag-pinned GitHub files and are hash-checked: the UD 2.17 TueCL treebanks, the KazNERD and KyrgyzNER releases and their README files, and the TIL corpus README. The audit in Exercise 1 also reads the book's UD 2.17 inventory (`tables/tbl-ud217-inventory/data.csv`), the NLLB-200 tokeniser (`NLLB_PATH`) and GlotLID (`GLOTLID_PATH`), and queries live services, so its live columns are dated observations, not pinned results. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's `observation` and `hub_count` helpers from `../code/remaining/resources.py`. Downloads are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

The cell defines the fields a plan must answer and a checker that lists every empty one. It is applied to a partly filled draft for an invented project; no real community, person or cost is implied.

In [ ]:
import collections, csv, datetime, difflib, hashlib, json, os, re, statistics, subprocess, sys, time
from pathlib import Path
import requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from resources import observation, hub_count

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD + "README.md": "2d56859aa35eadafb8bc560b375d5128627e3bab3517bc74377c7818905e3614",
    KAZNERD + "KazNERD/IOB2_train.txt": "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD + "KazNERD/IOB2_test.txt": "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER + "README.md": "c5beec0dd94156f5cdae4c5f7ffafeb23704f6fd1024ed4cb505cc8ef498a46c",
    TIL + "til_corpus/README.md": "da88f81cc7a23fb46425dcf236f7c89ad0f63663d2141d8fb7165ea8015308d8",
    UD.format("UD_Turkish-TueCL", "tr_tuecl-ud-test.conllu"): "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    UD.format("UD_Azerbaijani-TueCL", "az_tuecl-ud-test.conllu"): "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    UD.format("UD_Uzbek-TueCL", "uz_tuecl-ud-test.conllu"): "23257711196df0bbfaf5b25397d7c6102fbc8800ed83b15eb77342929ee0e469",
    UD.format("UD_Kyrgyz-TueCL", "ky_tuecl-ud-test.conllu"): "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

def read_conll(path):
    """KazNERD-style column file -> [(words, tags)]."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith("# "):
            c = line.split("\t") if "\t" in line else line.split()
            words.append(c[0]); tags.append(c[-1].strip())
    return sents

def show(obs):
    """One audit cell: the value, or the reason why there is none. Zero, not found, error and not checked stay distinct."""
    return {"observed": lambda o: str(o["value"]), "not_found": lambda o: "not found", "error": lambda o: "ERROR",
            "not_checked": lambda o: "not checked"}[obs["status"]](obs)
print("helpers loaded from code/remaining/resources.py:", observation.__name__, hub_count.__name__)

In [ ]:
# A desk template: every deliverable needs an owner, a permission basis, a withdrawal rule and a maintenance plan.
REQUIRED = {
    "partners": ["who represents the community for this project, and how was that agreed", "individual participants"],
    "deliverables": ["name", "who asked for it", "owner", "permission basis", "public/restricted/withheld",
                     "withdrawal: what changes, what cannot be recalled", "maintenance: who, how long", "pilot evidence for the effort"],
    "budget": ["recruitment and permissions", "annotation and adjudication", "payment of contributors",
               "development", "device testing", "maintenance after the project"],
    "decision points": ["when the plan can change", "who decides"],
}
def check(plan):
    """List empty or missing fields; never fills them in."""
    gaps = []
    for section, fields in REQUIRED.items():
        items = plan.get(section)
        if section == "deliverables":
            if not items: gaps.append("deliverables: none listed")
            for d in items or []:
                gaps += [f"deliverable '{d.get('name') or '?'}': {f}" for f in fields if not d.get(f)]
        else:
            gaps += [f"{section}: {f}" for f in fields if not (items or {}).get(f)]
    return gaps
# A partly filled draft for an invented project (no real community, person or budget figure is implied).
draft = {
    "partners": {"who represents the community for this project, and how was that agreed": "",
                 "individual participants": "speakers who volunteer after an information session"},
    "deliverables": [
        {"name": "searchable recording archive", "who asked for it": "", "owner": "partner archive",
         "permission basis": "individual consent form + archive agreement", "public/restricted/withheld": "restricted",
         "withdrawal: what changes, what cannot be recalled": "", "maintenance: who, how long": "", "pilot evidence for the effort": ""},
        {"name": "speech recogniser", "who asked for it": "", "owner": "", "permission basis": "",
         "public/restricted/withheld": "", "withdrawal: what changes, what cannot be recalled": "",
         "maintenance: who, how long": "", "pilot evidence for the effort": ""},
    ],
    "budget": {"payment of contributors": "per recorded hour, rate set with partners"},
    "decision points": {},
}
gaps = check(draft)
print(f"{len(gaps)} open items in the draft:")
for g in gaps: print("  -", g)

## Worked solution

Output of `experiments/ch17_exercises/run_ch17_exercises.py`:

```
19 open items in the draft:
  - partners: who represents the community for this project, and how was that agreed
  - deliverable 'searchable recording archive': who asked for it
  - deliverable 'searchable recording archive': withdrawal: what changes, what cannot be recalled
  - deliverable 'searchable recording archive': maintenance: who, how long
  - deliverable 'searchable recording archive': pilot evidence for the effort
  - deliverable 'speech recogniser': who asked for it
  - deliverable 'speech recogniser': owner
  - deliverable 'speech recogniser': permission basis
  - deliverable 'speech recogniser': public/restricted/withheld
  - deliverable 'speech recogniser': withdrawal: what changes, what cannot be recalled
  - deliverable 'speech recogniser': maintenance: who, how long
  - deliverable 'speech recogniser': pilot evidence for the effort
  - budget: recruitment and permissions
  - budget: annotation and adjudication
  - budget: development
  - budget: device testing
  - budget: maintenance after the project
  - decision points: when the plan can change
  - decision points: who decides
```

**The checker finds empty fields, not an approved plan.** It reports nineteen open items in the fictional draft, including representation, requested deliverables, withdrawal, maintenance, budgets and decision points. It does not test whether nonempty answers are true, complete or agreed.

Assign each decision to the appropriate participants, community representatives or responsible team members. Record how partners can choose among deliverables and revise the plan. Explain which future uses can stop after withdrawal and the limits of recalling copies already distributed.

No community was contacted. A real budget needs agreed rates, a timed pilot and maintenance commitments; plausible text in a template cannot supply that evidence.

## Your turn — extension

Write your own plan for a language you know, fill every field, and run the checker. Then mark each field with who would have to confirm it before the project could start.